In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from glob import glob
from tqdm import tqdm

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2'
outdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/csv'
donors = ['C29', 'C37', 'C38', 'C39']
mcdsdir = f'{projdir}/snmCT/mc/mcds/batch/chrom100k'
pbdir = f'{projdir}/snmCT/pseudobulk_allc'
maturedir = '/u/project/cluo/terencew/igvf/50_line/snmCT/pseudobulk_allc'
min_cov = 10

In [ ]:
nuclei = pd.read_csv(f'{outdir}/ipsc_le_se_all_nuclei.csv', sep='\t', index_col=0)
early = nuclei[nuclei['early_ipsc']]
nuclei.shape, early.shape

### pseudobulk 100kb CGN from per-nucleus MCDS (snmCT, as in the original allc pseudobulks)

In [ ]:
groups = {}
for t in ['alt', 'main']:
    for d in donors:
        orig = pd.read_csv(f'{pbdir}/txt/allc_table/ips_cluster_donor/IPS_{t}_{d}.txt', header=None)[0]
        orig = [x.split('/')[-1].split('.')[0] for x in orig]
        groups[f'orig_IPS_{t}_{d}'] = set(orig)
        x = early[(early['assay'] == 'snmCT') & (early['ips_subtype'] == f'IPS_{t}') & (early['line'] == d)]
        groups[f'IPS_{t}_{d}'] = set(x.index)
{k: len(v) for k, v in groups.items()}

In [ ]:
mc_sum = {k: 0 for k in groups}
cov_sum = {k: 0 for k in groups}
found = {k: 0 for k in groups}
for p in tqdm(sorted(glob(f'{mcdsdir}/*/chrom100k'))):
    ds = xr.open_zarr(p)
    cells = pd.Index(ds['cell'].values)
    for k, v in groups.items():
        m = cells.isin(list(v))
        if m.sum() == 0:
            continue
        da = ds['chrom100k_da'].sel(mc_type='CGN').isel(cell=np.where(m)[0]).sum('cell').values
        mc_sum[k] = mc_sum[k] + da[:, 0]
        cov_sum[k] = cov_sum[k] + da[:, 1]
        found[k] += m.sum()
    bins = ds['chrom100k'].values
{k: (len(groups[k]), found[k]) for k in groups}

In [ ]:
mc = pd.DataFrame(mc_sum, index=bins)
cov = pd.DataFrame(cov_sum, index=bins)
mc.shape

### sanity check: MCDS pseudobulk of the original nuclei vs the published allc pseudobulk

In [ ]:
def load_bins(path):
    x = pd.read_csv(path, sep='\t', header=None)
    x.index = [f'{a}_{int(b) // 100000}' for a, b in zip(x[0], x[1])]
    return x[~x.index.duplicated()]

checks = []
for d in donors:
    for t in ['alt', 'main']:
        b = load_bins(f'{pbdir}/bed/bins/ips_cluster_donor/IPS_{t}_{d}.chrom100k_CGN-Both.sparse.bed.gz')
        k = b.index.intersection(mc.index)
        checks.append([f'IPS_{t}_{d}', len(k), np.corrcoef(b.loc[k, 5], cov.loc[k, f'orig_IPS_{t}_{d}'])[0, 1], np.corrcoef(b.loc[k, 4], mc.loc[k, f'orig_IPS_{t}_{d}'])[0, 1]])
pd.DataFrame(checks, columns=['group', 'n_bins', 'r_cov', 'r_mc'])

### mature-iPSC references

In [ ]:
mature_samples = [f'{d}_IPS' for d in donors]
for s in mature_samples:
    b = load_bins(f'{maturedir}/bed/bins/donor_time/{s}.chrom100k_CGN-Both.sparse.bed.gz')
    mc[s] = b[4].reindex(mc.index).fillna(0)
    cov[s] = b[5].reindex(mc.index).fillna(0)
mc.shape

### correlation with >=10x in all 12 profiles, no fill (as in fig6h rerun)

In [ ]:
def summarize(sub_samples):
    samples = sub_samples + mature_samples
    keep = (cov[samples] >= min_cov).all(axis=1)
    frac = (mc[samples] / cov[samples]).loc[keep]
    frac = frac[~frac.index.str.match(r'chr[XYM]_')]
    corr = frac.corr()
    rows = []
    for t in ['alt', 'main']:
        names = [s for s in sub_samples if f'IPS_{t}_' in s]
        same = [corr.loc[n, f'{n[-3:]}_IPS'] for n in names]
        rows.append([t, len(frac), np.mean(same)] + same)
    return corr, pd.DataFrame(rows, columns=['subtype', 'n_bins', 'mean_same_donor'] + donors).set_index('subtype')

In [ ]:
orig_corr, orig_summary = summarize([f'orig_IPS_{t}_{d}' for t in ['alt', 'main'] for d in donors])
orig_summary

In [ ]:
early_corr, early_summary = summarize([f'IPS_{t}_{d}' for t in ['alt', 'main'] for d in donors])
early_summary

In [ ]:
early_corr.to_csv(f'{outdir}/ips_subtype_mcg_mature_corr_min10x.csv', sep='\t')
pd.concat({'all_nuclei_mcds': orig_summary, 'early_ipsc': early_summary}).to_csv(f'{outdir}/ips_subtype_mcg_mature_corr_summary.csv', sep='\t')
pd.DataFrame({'n_nuclei': {k: found[k] for k in groups}}).to_csv(f'{outdir}/ips_subtype_mcg_pseudobulk_counts.csv', sep='\t')

In [ ]:
!date